# Aerial Object Classification & Detection
## Notebook 2 — Custom CNN Model

In this notebook I'm building a Custom CNN from scratch to classify aerial images.
I want to see how well a basic CNN performs before moving to Transfer Learning.

**Expected outcome:** This will give us a baseline accuracy to compare against.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# checking if GPU is available — important for faster training
print('GPU Available:', tf.config.list_physical_devices('GPU'))

IMG_SIZE   = (128, 128)  # using 128 for faster training
BATCH_SIZE = 64
EPOCHS     = 20

print('Setup done!')

Copy Dataset to Local Storage

In [ ]:
import shutil

base = '/content/drive/MyDrive/'

if not os.path.exists('/content/dataset/train'):
    print('Copying dataset to local storage...')
    shutil.copytree(base + 'train-20260409T091319Z-3-001/train/', '/content/dataset/train')
    shutil.copytree(base + 'valid-20260409T091319Z-3-001/valid/', '/content/dataset/valid')
    shutil.copytree(base + 'test-20260409T091319Z-3-001/test/',  '/content/dataset/test')
    print('Done copying!')
else:
    print('Dataset already in local storage, skipping copy.')

# quick check
TRAIN_DIR = '/content/dataset/train/'
VALID_DIR = '/content/dataset/valid/'
TEST_DIR  = '/content/dataset/test/'

for split in ['train', 'valid', 'test']:
    for cls in ['bird', 'drone']:
        count = len(os.listdir(f'/content/dataset/{split}/{cls}'))
        print(f'{split}/{cls}: {count} images')

Data Generators


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VALID_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
print('Class mapping:', train_gen.class_indices)  # bird=0, drone=1

Build Custom CNN

I'm designing a CNN with 3 convolutional blocks.
Each block has Conv2D + BatchNormalization + MaxPooling + Dropout.
BatchNorm helps with faster training and Dropout prevents overfitting.

In [ ]:
from tensorflow.keras import layers, models

def build_custom_cnn():
    model = models.Sequential([

        # Block 1 — learning basic features like edges
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=(128, 128, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 2 — learning more complex features
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 3 — high level feature extraction
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Fully connected layers for classification
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),

        # Output — sigmoid for binary classification (bird or drone)
        layers.Dense(1, activation='sigmoid')
    ])
    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

Compile Model

In [ ]:
# using Adam optimizer and binary crossentropy since its a 2 class problem
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

os.makedirs('/content/drive/MyDrive/aerial_project/models/', exist_ok=True)
save_path = '/content/drive/MyDrive/aerial_project/models/custom_cnn_best.h5'

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(save_path, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1)]

Train the Model

In [ ]:
print('Starting training...\n')

history = cnn_model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print('\nTraining done!')

Plot Accuracy & Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato')
axes[0].set_title('Custom CNN — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('Custom CNN — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/aerial_project/models/cnn_training_plot.png')
plt.show()
print('Plot saved!')

Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

test_gen.reset()
preds       = cnn_model.predict(test_gen, verbose=1)
pred_labels = (preds > 0.5).astype(int).flatten()
true_labels = test_gen.classes

print('\nClassification Report:')
print(classification_report(true_labels, pred_labels,
                            target_names=['Bird', 'Drone']))

# confusion matrix shows where the model is making mistakes
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Bird', 'Drone'],
            yticklabels=['Bird', 'Drone'])
plt.title('Custom CNN — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/aerial_project/models/cnn_confusion_matrix.png')
plt.show()

In [ ]:
test_loss, test_acc, test_prec, test_rec = cnn_model.evaluate(test_gen, verbose=0)
f1 = 2 * (test_prec * test_rec) / (test_prec + test_rec + 1e-7)

print('=' * 40)
print('CUSTOM CNN — FINAL RESULTS')
print('=' * 40)
print(f'Test Accuracy  : {test_acc*100:.2f}%')
print(f'Test Precision : {test_prec*100:.2f}%')
print(f'Test Recall    : {test_rec*100:.2f}%')
print(f'F1 Score       : {f1*100:.2f}%')
print(f'Test Loss      : {test_loss:.4f}')
print('=' * 40)
print()